## Cell 0 — One-off: add cousin columns to existing snapshot table

Run once before the next snapshot. Safe to re-run.

In [ ]:
# One-off: add cousin metric columns to the existing snapshot table.
# Run this cell ONCE manually before the next scheduled snapshot run.
# Safe to re-run — ALTER TABLE ADD COLUMN is idempotent in Delta.
#
# sibling_coverage_3rd: % of gen-3 sibling lines with >=1 3rd cousin traced
# fill_rate_3rd:         3rd cousins found / 22 expected (UK avg / 8 branches) * 100
# fill_rate_4th:         4th cousins found / 131 expected (UK avg / 8 branches) * 100
# sibling_coverage_4th:  % of gen-4 sibling lines with >=1 4th cousin traced

for col, dtype in [
    ('sibling_coverage_3rd', 'DOUBLE'),
    ('fill_rate_3rd',         'DOUBLE'),
    ('fill_rate_4th',         'DOUBLE'),
    ('sibling_coverage_4th', 'DOUBLE'),
]:
    try:
        spark.sql(f'ALTER TABLE genealogy.gold_scores_snapshot_branch ADD COLUMN {col} {dtype}')
        print(f'Added column: {col} {dtype}')
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f'Column already exists (skipping): {col}')
        else:
            raise

print('ALTER TABLE complete.')

# Scores Snapshot

Takes a weekly snapshot of `gold_person_scores` and `gold_branch_scores` into
two Delta tables. Designed to be run as a standalone Databricks job every Sunday
at 3am UTC (after the 2am OCR run).

**Idempotent**: if a snapshot for today already exists, the cell skips cleanly.

**Tables created**:
- `genealogy.gold_scores_snapshot_person` — one row per person per snapshot
- `genealogy.gold_scores_snapshot_branch` — one row per branch per snapshot

**Retention**: no automatic pruning — rows accumulate indefinitely.
At 8 branches + ~6,000 people per week this is ~300k person-rows/year, trivially small.

**Schedule**: Databricks Jobs UI → New Job → Notebook task → this notebook.
Cron: `0 0 3 * * 0` (Quartz syntax, Sunday 3am UTC).

## Cell 1 — Create tables if not exist

In [ ]:
# Create snapshot tables if they don't already exist.
# snapshot_date is the Sunday the job runs — used as the partition key.

spark.sql("""
CREATE TABLE IF NOT EXISTS genealogy.gold_scores_snapshot_person (
    snapshot_date       DATE,
    person_gedcom_id    STRING,
    branch              STRING,
    proximity_label     STRING,
    completeness_score  DOUBLE,
    evidence_score      DOUBLE,
    overall_score       DOUBLE,
    story_ready         BOOLEAN,
    story_written       BOOLEAN
)
USING DELTA
COMMENT 'Weekly snapshot of gold_person_scores for trend tracking'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS genealogy.gold_scores_snapshot_branch (
    snapshot_date         DATE,
    branch                STRING,
    avg_completeness      DOUBLE,
    avg_evidence          DOUBLE,
    avg_overall           DOUBLE,
    stories_written       INT,
    stories_ready         INT,
    direct_ancestors      INT,
    ancestor_fill_score   DOUBLE,
    depth_score           DOUBLE,
    sibling_coverage_3rd  DOUBLE,
    fill_rate_3rd         DOUBLE,
    fill_rate_4th         DOUBLE,
    sibling_coverage_4th  DOUBLE
)
USING DELTA
COMMENT 'Weekly snapshot of gold_branch_scores for trend tracking'
""")

print('Tables ready.')

## Cell 2 — Check for existing snapshot today (idempotency guard)

In [ ]:
from datetime import date

today = date.today().isoformat()  # e.g. '2026-03-22'

existing = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM genealogy.gold_scores_snapshot_branch
    WHERE snapshot_date = '{today}'
""").collect()[0]['n']

if existing > 0:
    print(f'Snapshot for {today} already exists ({existing} branch rows). Skipping.')
    dbutils.notebook.exit('already_done')
else:
    print(f'No snapshot for {today} found. Proceeding.')

## Cell 3 — Snapshot gold_scores_snapshot_person

In [ ]:
person_result = spark.sql(f"""
INSERT INTO genealogy.gold_scores_snapshot_person
SELECT
    DATE('{today}')      AS snapshot_date,
    person_gedcom_id,
    branch,
    proximity_label,
    ROUND(completeness_score, 4)  AS completeness_score,
    ROUND(evidence_score,     4)  AS evidence_score,
    ROUND(overall_score,      4)  AS overall_score,
    COALESCE(story_ready,   FALSE) AS story_ready,
    COALESCE(story_written, FALSE) AS story_written
FROM genealogy.gold_person_scores
""")

person_count = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM genealogy.gold_scores_snapshot_person
    WHERE snapshot_date = '{today}'
""").collect()[0]['n']

print(f'Person snapshot done: {person_count} rows written for {today}.')

## Cell 4 — Snapshot gold_scores_snapshot_branch

In [ ]:
# gold_branch_scores already has the aggregated columns we need.
# direct_ancestors: count of people with ancestral_proximity = 0 in this branch.
# ancestor_fill_score: direct ancestors found at gen 3-8 as % of 63 per-branch slots.
# depth_score: max generation depth as % of 21-generation ceiling.
# sibling_coverage_3rd/4th: % of sibling lines with >=1 cousin traced at that degree.
# fill_rate_3rd/4th: cousins found vs UK average expectation per branch.
#
# Note: gold_generation_depth.person_id holds the same values as
# gold_person_branch.person_gedcom_id — column rename is pending.

branch_result = spark.sql(f"""
INSERT INTO genealogy.gold_scores_snapshot_branch
WITH tree_shape AS (
    SELECT
        gpb.branch,
        COUNT(DISTINCT CASE WHEN ggd.generation_depth BETWEEN 3 AND 8
                            THEN ggd.person_id END)          AS ancestors_found,
        MAX(ggd.generation_depth)                             AS max_depth
    FROM genealogy.gold_generation_depth ggd
    JOIN genealogy.gold_person_branch gpb
        ON gpb.person_gedcom_id = ggd.person_id
    WHERE ggd.generation_depth >= 3
    GROUP BY gpb.branch
),
cousin3 AS (
    -- 3rd cousin metrics: siblings of gen-3 ancestors and their descendants
    WITH gen4_ancestors AS (
        SELECT person_id AS gggg_id, path[3] AS direct_gen3_id
        FROM genealogy.gold_generation_depth WHERE generation_depth = 4
    ),
    gen4_families AS (
        SELECT g.gggg_id, g.direct_gen3_id, f.family_gedcom_id
        FROM gen4_ancestors g
        JOIN genealogy.silver_family f
            ON f.husband_gedcom_id = g.gggg_id OR f.wife_gedcom_id = g.gggg_id
    ),
    collateral_gen3 AS (
        SELECT DISTINCT fc.child_gedcom_id AS collateral_id, gf.gggg_id
        FROM gen4_families gf
        JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = gf.family_gedcom_id
        WHERE fc.child_gedcom_id != gf.direct_gen3_id
    ),
    collateral_gen3_children AS (
        SELECT DISTINCT fc.child_gedcom_id AS child_id,
                        c3.collateral_id AS sibling_id, c3.gggg_id
        FROM collateral_gen3 c3
        JOIN genealogy.silver_family f
            ON f.husband_gedcom_id = c3.collateral_id OR f.wife_gedcom_id = c3.collateral_id
        JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
    ),
    third_cousins AS (
        SELECT DISTINCT fc.child_gedcom_id AS third_cousin_id,
                        cgc.sibling_id, cgc.gggg_id
        FROM collateral_gen3_children cgc
        JOIN genealogy.silver_family f
            ON f.husband_gedcom_id = cgc.child_id OR f.wife_gedcom_id = cgc.child_id
        JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
    )
    SELECT gpb.branch,
           COUNT(DISTINCT c3.collateral_id) AS known_siblings,
           COUNT(DISTINCT CASE WHEN tc.third_cousin_id IS NOT NULL
                               THEN c3.collateral_id END) AS siblings_with_cousins,
           COUNT(DISTINCT tc.third_cousin_id) AS cousins_found
    FROM collateral_gen3 c3
    JOIN genealogy.gold_person_branch gpb ON gpb.person_gedcom_id = c3.collateral_id
    LEFT JOIN third_cousins tc ON tc.sibling_id = c3.collateral_id
    GROUP BY gpb.branch
),
cousin4 AS (
    -- 4th cousin metrics: siblings of gen-4 ancestors and their descendants
    WITH gen5_ancestors AS (
        SELECT person_id AS ggggg_id, path[4] AS direct_gen4_id
        FROM genealogy.gold_generation_depth WHERE generation_depth = 5
    ),
    gen5_families AS (
        SELECT g.ggggg_id, g.direct_gen4_id, f.family_gedcom_id
        FROM gen5_ancestors g
        JOIN genealogy.silver_family f
            ON f.husband_gedcom_id = g.ggggg_id OR f.wife_gedcom_id = g.ggggg_id
    ),
    collateral_gen4 AS (
        SELECT DISTINCT fc.child_gedcom_id AS collateral_id, gf.ggggg_id
        FROM gen5_families gf
        JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = gf.family_gedcom_id
        WHERE fc.child_gedcom_id != gf.direct_gen4_id
    ),
    c4g3 AS (
        SELECT DISTINCT fc.child_gedcom_id AS collateral_id,
                        c4.ggggg_id, c4.collateral_id AS sibling_id
        FROM collateral_gen4 c4
        JOIN genealogy.silver_family f
            ON f.husband_gedcom_id = c4.collateral_id OR f.wife_gedcom_id = c4.collateral_id
        JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
    ),
    c4g2 AS (
        SELECT DISTINCT fc.child_gedcom_id AS collateral_id,
                        c.ggggg_id, c.sibling_id
        FROM c4g3 c
        JOIN genealogy.silver_family f
            ON f.husband_gedcom_id = c.collateral_id OR f.wife_gedcom_id = c.collateral_id
        JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
    ),
    c4g1 AS (
        SELECT DISTINCT fc.child_gedcom_id AS collateral_id,
                        c.ggggg_id, c.sibling_id
        FROM c4g2 c
        JOIN genealogy.silver_family f
            ON f.husband_gedcom_id = c.collateral_id OR f.wife_gedcom_id = c.collateral_id
        JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
    ),
    fourth_cousins AS (
        SELECT DISTINCT fc.child_gedcom_id AS fourth_cousin_id,
                        c.sibling_id, c.ggggg_id
        FROM c4g1 c
        JOIN genealogy.silver_family f
            ON f.husband_gedcom_id = c.collateral_id OR f.wife_gedcom_id = c.collateral_id
        JOIN genealogy.silver_family_child fc ON fc.family_gedcom_id = f.family_gedcom_id
    )
    SELECT gpb.branch,
           COUNT(DISTINCT c4.collateral_id) AS known_siblings,
           COUNT(DISTINCT CASE WHEN fc4.fourth_cousin_id IS NOT NULL
                               THEN c4.collateral_id END) AS siblings_with_cousins,
           COUNT(DISTINCT fc4.fourth_cousin_id) AS cousins_found
    FROM collateral_gen4 c4
    JOIN genealogy.gold_person_branch gpb ON gpb.person_gedcom_id = c4.collateral_id
    LEFT JOIN fourth_cousins fc4 ON fc4.sibling_id = c4.collateral_id
    GROUP BY gpb.branch
)
SELECT
    DATE('{today}')                            AS snapshot_date,
    bs.branch,
    ROUND(bs.avg_completeness, 4)              AS avg_completeness,
    ROUND(bs.avg_evidence,     4)              AS avg_evidence,
    ROUND(bs.avg_overall,      4)              AS avg_overall,
    COALESCE(bs.stories_written, 0)            AS stories_written,
    COALESCE(bs.stories_ready,   0)            AS stories_ready,
    COALESCE(da.direct_ancestors, 0)           AS direct_ancestors,
    ROUND(COALESCE(ts.ancestors_found, 0)
          / 63.0 * 100, 4)                    AS ancestor_fill_score,
    ROUND(COALESCE(ts.max_depth, 0)
          / 21.0 * 100, 4)                    AS depth_score,
    ROUND(COALESCE(c3.siblings_with_cousins, 0)
          / NULLIF(c3.known_siblings, 0) * 100, 4) AS sibling_coverage_3rd,
    ROUND(COALESCE(c3.cousins_found, 0)
          / 22.0 * 100, 4)                    AS fill_rate_3rd,
    ROUND(COALESCE(c4.cousins_found, 0)
          / 131.0 * 100, 4)                   AS fill_rate_4th,
    ROUND(COALESCE(c4.siblings_with_cousins, 0)
          / NULLIF(c4.known_siblings, 0) * 100, 4) AS sibling_coverage_4th
FROM genealogy.gold_branch_scores bs
LEFT JOIN (
    SELECT branch, COUNT(*) AS direct_ancestors
    FROM genealogy.gold_person_scores
    WHERE ancestral_proximity = 0
    GROUP BY branch
) da ON da.branch = bs.branch
LEFT JOIN tree_shape ts ON ts.branch = bs.branch
LEFT JOIN cousin3 c3 ON c3.branch = bs.branch
LEFT JOIN cousin4 c4 ON c4.branch = bs.branch
""")

## Cell 5 — Verify: show latest two snapshots

In [ ]:
display(spark.sql("""
    SELECT snapshot_date, branch, avg_completeness, avg_evidence, avg_overall,
           stories_written, stories_ready,
           ancestor_fill_score, depth_score,
           sibling_coverage_3rd, fill_rate_3rd, fill_rate_4th
    FROM genealogy.gold_scores_snapshot_branch
    WHERE snapshot_date IN (
        SELECT DISTINCT snapshot_date
        FROM genealogy.gold_scores_snapshot_branch
        ORDER BY snapshot_date DESC
        LIMIT 2
    )
    ORDER BY snapshot_date DESC, branch
"""))